# 🎙️ Fine-tune giọng nói Piper TTS từ dataset của bạn (bản Kaggle)

Notebook này nhận **dataset đã thu bằng VoiceRecorder** (metadata.csv + các file .wav), fine-tune từ checkpoint tiếng Việt có sẵn (`vi_VN-vais1000-medium`), rồi xuất ra `.onnx` + `.onnx.json` để dùng với Piper.

**Đây là bản chạy trên Kaggle** (không dùng Google Drive) — dùng khi Colab hết/đòi nâng cấp trả phí để dùng GPU. Kaggle cho **30 giờ GPU T4/tuần miễn phí**.

**Trước khi chạy:**
1. **Xác minh số điện thoại** (chỉ 1 lần): ảnh đại diện góc phải → Settings → Phone Verification. Không làm bước này thì Kaggle ẩn hẳn nút GPU/Internet.
2. Bấm **Settings** (thanh tab trên cùng của notebook, hoặc bảng bên phải khi đang ở chế độ Edit) → **Accelerator → GPU T4 x2**, và **Internet → ON**.
3. Nén thư mục `dataset` (nằm cạnh file `.exe` đã build, ví dụ `...\bin\Release\net9.0-windows\dataset\`) thành `dataset.zip`.
4. Vào tab **Input** (bên phải) → **Upload → New Dataset** → chọn `dataset.zip`. Kaggle thường **tự động giải nén** file zip khi tạo dataset, nên trong tab Input bạn sẽ thấy sẵn các thư mục con (ví dụ `01_co_ban/`, `02_cau_hoi_cam_than/`...) chứ không còn thấy `dataset.zip` nữa — đó là bình thường. Bấm nút copy cạnh tên dataset (icon 📋 trong tab Input) để lấy đúng đường dẫn dạng `/kaggle/input/<ten-dataset>`, rồi dán vào `DATASET_ZIP` ở Bước 0 bên dưới (code đã tự nhận diện được cả 2 trường hợp: thư mục hoặc file zip).
5. Chạy **lần lượt từng ô từ trên xuống dưới** (bấm ▶), hoặc **Run → Run All**. Mỗi phiên Kaggle mới đều trắng tinh (giống Colab) — phải chạy lại từ Bước 0.

**Lưu ý về license:** notebook này dùng bộ code huấn luyện gốc của `rhasspy/piper` (MIT) vì nó tương thích trực tiếp với checkpoint `vais1000-medium`. Bản kế nhiệm `OHF-Voice/piper1-gpl` (GPL-3.0) hiện cộng đồng báo là chưa nạp ổn định các checkpoint cũ.

## Bước 0: Thiết lập tham số

In [ ]:
#@markdown ### Nguồn dữ liệu
#@markdown Đường dẫn tới dataset đã upload qua tab "Input" (xem huong dan phia tren de lay dung duong dan).
#@markdown Co the la file `dataset.zip` HOAC thang thu muc dataset (Kaggle hay tu giai nen file zip ban upload thanh thu muc san, ca 2 truong hop deu duoc code tu nhan dien):
DATASET_ZIP = "/kaggle/input/ten-dataset-cua-ban"  #@param {type:"string"}

#@markdown Tên giọng nói (chỉ chữ/số/gạch dưới, không dấu, không khoảng trắng):
VOICE_NAME = "giong_cua_toi"  #@param {type:"string"}

#@markdown (Tuỳ chọn) Nếu đang train tiếp từ 1 phiên Kaggle trước đó (xem "Bước 5c" bên dưới) — dán đường dẫn tới thư mục output của phiên trước (dạng `/kaggle/input/<notebook-output-cua-ban>/<VOICE_NAME>_train`). Để trống nếu train mới từ đầu:
RESUME_INPUT_DIR = ""  #@param {type:"string"}

#@markdown Batch size (Kaggle T4 thường hợp 8-16; giảm xuống nếu bị lỗi hết bộ nhớ GPU):
BATCH_SIZE = 12  #@param {type:"integer"}

#@markdown Train thêm bao nhiêu epoch nữa tính từ checkpoint hiện tại:
EXTRA_EPOCHS = 1000  #@param {type:"integer"}

TRAIN_ROOT = '/kaggle/working/piper_training'
print('TRAIN_ROOT (noi luu tien do, nam trong /kaggle/working, dung luong toi da ~19.5GB moi phien):', TRAIN_ROOT)
if RESUME_INPUT_DIR:
    print('Se thu tiep tuc tu checkpoint cu tai:', RESUME_INPUT_DIR)

## Bước 1: Kết nối dữ liệu
Lấy dữ liệu từ Input đã đính kèm — tự nhận diện dù là file `dataset.zip` hay thư mục đã được Kaggle tự giải nén sẵn (không dùng Google Drive trên Kaggle).

In [ ]:
import os

assert os.path.exists(DATASET_ZIP), f'Khong tim thay: {DATASET_ZIP}. Kiem tra lai duong dan trong o Buoc 0 (xem tab Input ben phai de lay dung duong dan).'

os.makedirs('/kaggle/working/dataset_raw', exist_ok=True)
os.makedirs(TRAIN_ROOT, exist_ok=True)

if os.path.isdir(DATASET_ZIP):
    print('DATASET_ZIP la mot thu muc (Kaggle da tu giai nen san khi upload) -> copy truc tiep.')
    !cp -r "{DATASET_ZIP}"/. /kaggle/working/dataset_raw/
else:
    print('DATASET_ZIP la file nen -> giai nen.')
    !unzip -q -o "{DATASET_ZIP}" -d /kaggle/working/dataset_raw

print('Da xong buoc lay du lieu. Noi dung:')
!find /kaggle/working/dataset_raw -maxdepth 4 -type d
print()
print('So file tim thay:')
!find /kaggle/working/dataset_raw -type f | wc -l

## Bước 1b: Gộp các bộ câu (nếu bạn thu bằng nhiều "Kịch bản" khác nhau) thành 1 dataset duy nhất

VoiceRecorder lưu mỗi bộ câu (kịch bản) vào một thư mục con riêng (`<ten_bo_cau>/wavs` + `<ten_bo_cau>/metadata.csv`), mỗi bộ đều đánh số file lại từ `0001` — nên không thể ghép trực tiếp, phải đổi số lại cho không trùng. Ô dưới tự dò và gộp hết lại.

In [ ]:
import shutil

combined_wavs = '/kaggle/working/combined_dataset/wavs'
combined_meta = '/kaggle/working/combined_dataset/metadata.csv'
os.makedirs(combined_wavs, exist_ok=True)

# tim tat ca cac thu muc co ca wavs/ va metadata.csv ben trong (moi thu muc la 1 bo cau)
dataset_sets = []
for root, dirs, filenames in os.walk('/kaggle/working/dataset_raw'):
    if 'metadata.csv' in filenames and 'wavs' in dirs:
        dataset_sets.append(root)

if len(dataset_sets) == 0:
    print('KHONG TIM THAY BO DU LIEU NAO. Toan bo file da giai nen:')
    !find /kaggle/working/dataset_raw -type f
    print()
    print('=> Kiem tra: (1) da Add Input dung dataset chua (thu muc hoac file .zip nen tu thu muc "dataset" canh file .exe),')
    print('   (2) trong app VoiceRecorder dong chu "Da thu: X" co X > 0 khong (X=0 nghia la chua luu duoc cau nao, se khong co metadata.csv).')

assert len(dataset_sets) > 0, 'Khong tim thay bo du lieu nao (can co wavs/ va metadata.csv). Xem log phia tren de biet ly do.'
print(f'Tim thay {len(dataset_sets)} bo cau:')
for s in dataset_sets:
    print(' -', s)

global_id = 1
rows = []
for set_dir in sorted(dataset_sets):
    meta_path = os.path.join(set_dir, 'metadata.csv')
    wavs_dir = os.path.join(set_dir, 'wavs')
    with open(meta_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '|' not in line:
                continue
            old_id, text = line.split('|', 1)
            src_wav = os.path.join(wavs_dir, old_id.strip() + '.wav')
            if not os.path.exists(src_wav):
                continue
            new_id = f'{global_id:04d}'
            shutil.copy(src_wav, os.path.join(combined_wavs, new_id + '.wav'))
            rows.append(f'{new_id}|{text}')
            global_id += 1

assert len(rows) > 0, 'Tim thay thu muc bo cau nhung metadata.csv rong hoac khong khop voi file .wav nao ca.'

with open(combined_meta, 'w', encoding='utf-8') as f:
    f.write('\n'.join(rows) + '\n')

print(f'Da gop xong: {len(rows)} cau thoai vao /kaggle/working/combined_dataset')

## Bước 2: Cài đặt môi trường huấn luyện Piper
(Chạy 1 lần, mất khoảng 3-5 phút)

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/rhasspy/piper.git /kaggle/working/piper

# requirements.txt goc cua piper ghim nhieu phien ban qua cu, khong con phu hop voi
# moi truong Colab hien tai (Python 3.12 + torch 2.11 co san). Vá lai truoc khi cai:
# 1. piper-phonemize~=1.1.0  -> dung ban thay the 'piper-phonemize-fix'
# 2. pytorch-lightning~=1.7.0 -> ghim sang 1.9.5 (moi hon nhung van gan API cu)
# 3. torch<2,>=1.11.0        -> bo han rang buoc, dung luon torch co san tren Colab
# 4. cython>=0.29.0,<1       -> ghim sang Cython>=3 de tuong thich Python 3.12
!pip install -q piper-phonemize-fix

import re
req_path = '/kaggle/working/piper/src/python/requirements.txt'
setup_path = '/kaggle/working/piper/src/python/setup.py'

with open(req_path, encoding='utf-8') as f:
    content = f.read()
content = re.sub(r'(?im)^.*piper-phonemize.*\n?', '', content)
content = re.sub(r'(?im)^pytorch-lightning.*\n?', 'pytorch-lightning==1.9.5\n', content)
content = re.sub(r'(?im)^torch\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^torchvision\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^torchaudio\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^cython.*\n?', 'cython>=3.0.0\n', content)
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(content)
print('Da sua requirements.txt xong.')

with open(setup_path, encoding='utf-8') as f:
    setup_content = f.read()
patched_setup = re.sub(r'(?im)^.*piper-phonemize.*\n?', '', setup_content)
if patched_setup != setup_content:
    with open(setup_path, 'w', encoding='utf-8') as f:
        f.write(patched_setup)
    print('Da go dong piper-phonemize trong setup.py')

!pip install -q -U 'cython>=3.0.0'

%cd /kaggle/working/piper/src/python
!pip install -q -e .
!rm -f /kaggle/working/piper/src/python/piper_train/vits/monotonic_align/core.c
!bash build_monotonic_align.sh

ok = True
try:
    import piper_phonemize
    print('piper_phonemize hoat dong binh thuong.')
except ImportError as e:
    ok = False
    print('LOI: piper_phonemize chua import duoc:', e)

try:
    import torch
    import pytorch_lightning
    import piper_train
    print('torch', torch.__version__, '| pytorch_lightning', pytorch_lightning.__version__, '| piper_train da cai dat thanh cong.')
    print('GPU kha dung:', torch.cuda.is_available())
except ImportError as e:
    ok = False
    print('LOI: chua cai dat duoc:', e)

import glob
so_files = glob.glob('/kaggle/working/piper/src/python/piper_train/vits/monotonic_align/**/*.so', recursive=True)
print('File .so tim thay:', so_files if so_files else '(khong co)')
monotonic_align_ok = len(so_files) > 0

# kiem tra thuc te bang cach import module dung nhu code piper se import
try:
    from piper_train.vits import monotonic_align
    print('Import piper_train.vits.monotonic_align: OK')
    monotonic_align_ok = True
except Exception as e:
    print('Import piper_train.vits.monotonic_align: LOI ->', e)
    monotonic_align_ok = False

ok = ok and monotonic_align_ok
print('Cai dat xong.' if ok else 'CAI DAT CHUA HOAN TAT - xem loi phia tren.')

## Bước 2b: Vá lỗi "Weights only load failed" (torch 2.6+)
Từ PyTorch 2.6, `torch.load` mặc định bật `weights_only=True`, khiến các checkpoint cũ (như `vais1000-medium`) load bị lỗi `UnpicklingError`. Ô dưới vá lại `lightning_fabric` để luôn load với `weights_only=False` (an toàn vì checkpoint lấy từ kho chính thức `rhasspy/piper-checkpoints`).

In [ ]:
import re
import lightning_fabric.utilities.cloud_io as cloud_io

cloud_io_path = cloud_io.__file__
with open(cloud_io_path, encoding="utf-8") as f:
    content = f.read()

patched = re.sub(
    r"torch\.load\(f, map_location=map_location\)",
    "torch.load(f, map_location=map_location, weights_only=False)",
    content,
)

if patched != content:
    with open(cloud_io_path, "w", encoding="utf-8") as f:
        f.write(patched)
    print("Da va xong:", cloud_io_path)
else:
    if "weights_only=False" in content:
        print("Da duoc va tu truoc do, khong can lam gi them.")
    else:
        print("KHONG TIM THAY DONG CAN VA. Duong dan file:", cloud_io_path)
        print("Hay bao loi nay lai, co the phien ban pytorch_lightning da doi cach load checkpoint.")

## Bước 3: Tiền xử lý dataset
Chuẩn hoá audio (resample 22050Hz, mono) và sinh cache phoneme từ `combined_dataset`.

In [ ]:
PREPROCESS_DIR = f'{TRAIN_ROOT}/{VOICE_NAME}_train'
os.makedirs(PREPROCESS_DIR, exist_ok=True)

%cd /kaggle/working/piper/src/python
!python3 -m piper_train.preprocess \
  --language vi \
  --input-dir /kaggle/working/combined_dataset \
  --output-dir "{PREPROCESS_DIR}" \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

dataset_jsonl = os.path.join(PREPROCESS_DIR, 'dataset.jsonl')
if os.path.exists(dataset_jsonl):
    print('Preprocess THANH CONG, du lieu luu tai:', PREPROCESS_DIR)
else:
    print('Preprocess CHUA XONG (bi dung giua chung hoac loi). Chay lai chinh o nay, KHONG dung giua chung.')

## Bước 4: Tải checkpoint tiếng Việt `vais1000-medium` để fine-tune
(Tự tìm đúng tên file checkpoint trên Hugging Face, không cần tự tra tên file)

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
import re

REPO_ID = 'rhasspy/piper-checkpoints'
CKPT_PREFIX = 'vi/vi_VN/vais1000/medium/'

api = HfApi()
files_list = api.list_repo_files(REPO_ID, repo_type='dataset')
ckpt_candidates = [f for f in files_list if f.startswith(CKPT_PREFIX) and f.endswith('.ckpt')]
assert ckpt_candidates, 'Khong tim thay checkpoint vais1000-medium tren Hugging Face.'
ckpt_remote_path = ckpt_candidates[0]
print('Checkpoint:', ckpt_remote_path)

pretrained_ckpt = hf_hub_download(repo_id=REPO_ID, filename=ckpt_remote_path, repo_type='dataset')
print('Da tai ve:', pretrained_ckpt)

m = re.search(r'epoch=(\d+)', ckpt_remote_path)
pretrained_epoch = int(m.group(1)) if m else 0
print('Epoch cua checkpoint goc:', pretrained_epoch)

## Bước 5: Fine-tune
Ô này tự phát hiện nếu bạn đã train dở từ trước (checkpoint đã lưu trong `TRAIN_ROOT`) để train tiếp; nếu chưa thì bắt đầu từ checkpoint `vais1000-medium` gốc.

In [ ]:
import glob, shutil

TRAINING_DIR = PREPROCESS_DIR  # piper luu lightning_logs/checkpoint ngay trong day

# Neu co RESUME_INPUT_DIR (tiep tuc tu 1 phien Kaggle truoc), copy checkpoint cu vao truoc khi quet
if RESUME_INPUT_DIR:
    resume_src = os.path.join(RESUME_INPUT_DIR, 'lightning_logs')
    resume_dst = os.path.join(TRAINING_DIR, 'lightning_logs')
    if os.path.exists(resume_src) and not os.path.exists(resume_dst):
        shutil.copytree(resume_src, resume_dst)
        print('Da copy checkpoint tu phien Kaggle truoc vao:', resume_dst)
    elif not os.path.exists(resume_src):
        print('CANH BAO: khong tim thay lightning_logs trong RESUME_INPUT_DIR:', resume_src)

existing_ckpts = glob.glob(f'{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt')

if existing_ckpts:
    existing_ckpts.sort(key=os.path.getmtime)
    resume_ckpt = existing_ckpts[-1]
    m = re.search(r'epoch=(\d+)', os.path.basename(resume_ckpt))
    start_epoch = int(m.group(1)) if m else pretrained_epoch
    print('Phat hien tien do da train truoc do, tiep tuc tu:', resume_ckpt)
else:
    resume_ckpt = pretrained_ckpt
    start_epoch = pretrained_epoch
    print('Bat dau fine-tune moi tu checkpoint vais1000-medium goc.')

max_epochs = start_epoch + EXTRA_EPOCHS
print(f'Se train den epoch {max_epochs} (hien tai dang o epoch {start_epoch}).')

In [ ]:
%cd /kaggle/working/piper/src/python
!python3 -m piper_train \
  --dataset-dir "{TRAINING_DIR}" \
  --accelerator gpu \
  --devices 1 \
  --batch-size {BATCH_SIZE} \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --quality medium \
  --checkpoint-epochs 1 \
  --precision 32 \
  --max_epochs {max_epochs} \
  --resume_from_checkpoint "{resume_ckpt}"

## Bước 5b: Dọn checkpoint cũ (tiết kiệm dung lượng Drive)
Mỗi lần restart, Lightning tạo thư mục `version_x` mới và checkpoint chất chồng qua các thư mục. Ô này gom tất cả checkpoint trong mọi `version_*`, chỉ giữ lại **N checkpoint có epoch mới nhất**, xoá phần còn lại để đỡ tốn dung lượng Drive (mỗi file ~800MB+). Chạy ô này sau khi train xong (hoặc bất cứ lúc nào muốn dọn), **không ảnh hưởng** tới việc resume vì Bước 5 luôn tự tìm checkpoint epoch cao nhất.

In [ ]:
import glob, os, re

KEEP_LATEST = 3  # so luong checkpoint moi nhat muon giu lai

all_ckpts = glob.glob(f"{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt")

def _epoch_of(path):
    m = re.search(r"epoch=(\d+)", os.path.basename(path))
    return int(m.group(1)) if m else -1

all_ckpts.sort(key=_epoch_of)

if len(all_ckpts) <= KEEP_LATEST:
    print(f"Chi co {len(all_ckpts)} checkpoint, chua can don dep.")
else:
    to_delete = all_ckpts[:-KEEP_LATEST]
    to_keep = all_ckpts[-KEEP_LATEST:]
    freed = 0
    for p in to_delete:
        freed += os.path.getsize(p)
        os.remove(p)
    print(f"Da xoa {len(to_delete)} checkpoint cu, giai phong {freed / (1024**3):.2f} GB.")
    print("Con lai (giu lai):")
    for p in to_keep:
        print(" -", p)

**Nếu phiên Kaggle bị hết giờ/ngắt giữa lúc train:** xem mục "Bước 5c" ngay dưới đây để lưu và tiếp tục ở phiên mới.

**Muốn train thêm nữa sau khi đã xong?** Chỉ cần tăng `EXTRA_EPOCHS` ở Bước 0 rồi chạy lại 2 ô của Bước 5.

## Bước 5c: Tiếp tục train ở phiên Kaggle mới (khi hết 12 tiếng/phiên hoặc hết 30 giờ GPU/tuần)

Kaggle **không giữ `/kaggle/working` giữa các phiên** như Drive bên Colab — mỗi phiên mới đều trắng tinh. Cách train tiếp:

1. Ở phiên hiện tại (trước khi hết giờ, hoặc ngay sau khi bị ngắt vẫn còn dữ liệu tạm): bấm **Save Version → Save & Run All (Commit)** ở góc trên bên phải. Đợi commit xong — toàn bộ `/kaggle/working` (bao gồm checkpoint đang train dở) sẽ được lưu vào **Output** của phiên đó.
2. Vào tab **Output** của notebook (sau khi commit xong) để xác nhận thấy đường dẫn `piper_training/<VOICE_NAME>_train/lightning_logs/...`.
3. Mở lại notebook này (hoặc tạo bản mới) → tab **Input** → **Add Input** → tìm chính notebook này (Kaggle cho phép gắn Output của 1 notebook làm Input cho notebook khác/phiên khác) → Add.
4. Kaggle sẽ cho biết đường dẫn dạng `/kaggle/input/<ten-notebook>/piper_training/<VOICE_NAME>_train` — dán đúng đường dẫn này vào `RESUME_INPUT_DIR` ở **Bước 0**.
5. Chạy lại **từ Bước 0** → Bước 1 → Bước 1b → Bước 2 → Bước 2b (bắt buộc phải làm lại các bước này vì `/kaggle/working` đã trắng tinh) → **Bước 5**. Ở Bước 5, notebook sẽ tự copy checkpoint từ `RESUME_INPUT_DIR` vào và train tiếp — không mất tiến độ.

In [ ]:
# (Khong can dung o nay tren ban Kaggle — da thay bang co che RESUME_INPUT_DIR o Buoc 0 + Buoc 5.
#  Giu lai o nay chi de ghi chu, khong co gi de chay.)
print('Xem huong dan "Buoc 5c" phia tren de train tiep o phien Kaggle moi.')

In [ ]:
# (Khong can dung o nay tren ban Kaggle — da thay bang co che RESUME_INPUT_DIR o Buoc 0 + Buoc 5.)
pass

## Bước 6: Xuất ra ONNX và nghe thử

In [ ]:
ckpts = glob.glob(f'{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt')
ckpts.sort(key=os.path.getmtime)
final_ckpt = ckpts[-1]
print('Dung checkpoint moi nhat:', final_ckpt)

onnx_path = f'/kaggle/working/{VOICE_NAME}.onnx'
%cd /kaggle/working/piper/src/python
!python3 -m piper_train.export_onnx "{final_ckpt}" "{onnx_path}"

config_path = f'{onnx_path}.json'
shutil.copy(f'{TRAINING_DIR}/config.json', config_path)
print('Da xuat xong:', onnx_path, 'va', config_path)

In [ ]:
!pip install -q piper-tts

test_text = 'Xin chào, đây là giọng nói được huấn luyện từ dữ liệu của bạn.'  #@param {type:"string"}
!echo "{test_text}" | piper --model "{onnx_path}" --output_file /kaggle/working/test_output.wav

from IPython.display import Audio, display
display(Audio('/kaggle/working/test_output.wav'))

## Bước 7: Lưu file cuối cùng
Trên Kaggle, mọi thứ trong `/kaggle/working` chỉ thực sự được lưu lại khi bạn **Commit** (Save Version).

In [ ]:
FINAL_DIR = '/kaggle/working/ket_qua'
os.makedirs(FINAL_DIR, exist_ok=True)
shutil.copy(onnx_path, f'{FINAL_DIR}/{VOICE_NAME}.onnx')
shutil.copy(config_path, f'{FINAL_DIR}/{VOICE_NAME}.onnx.json')
print('Da sao chep vao:', FINAL_DIR)
print()
print('QUAN TRONG: bam "Save Version" -> "Save & Run All (Commit)" o goc tren ben phai de luu that su.')
print('Sau khi commit xong, vao tab Output, mo thu muc ket_qua/ va tai 2 file .onnx + .onnx.json ve may.')
print('2 file nay dung truc tiep voi app .NET (piper voice).')

## 🔁 Train theo nhiều đợt (thu thêm dữ liệu rồi train tiếp)

Không cần thu hết một lần — thu một phần, train thử, rồi quay lại thu thêm và train tiếp:

1. Thu một phần câu bằng VoiceRecorder (ví dụ 100-150 câu đầu).
2. Nén `dataset` → `dataset.zip` rồi upload làm Input (Kaggle có thể tự giải nén thành thư mục, không sao cả), chạy notebook này từ đầu (đợt train 1), cuối cùng **Commit** để lưu.
3. Thu thêm câu (không quan trọng thu bộ cũ hay bộ mới).
4. Nén lại **toàn bộ** thư mục `dataset` thành `dataset.zip` mới, upload làm Input mới (hoặc cập nhật Input cũ) — dù Kaggle hiển thị dạng file `.zip` hay đã tự giải nén thành thư mục đều dùng được.
5. Mở notebook, Add Input = Output của lần Commit trước (xem "Bước 5c"), dán vào `RESUME_INPUT_DIR`, cập nhật `DATASET_ZIP` nếu dùng bộ dữ liệu mới (copy đúng đường dẫn từ tab Input).
6. Chạy lại từ Bước 0 đến Bước 5 — notebook tự nhận checkpoint cũ và train tiếp, không mất tiến độ.

Lặp lại bước 3-6 bao nhiêu đợt tuỳ ý. Luôn giữ `VOICE_NAME` giống nhau giữa các đợt.

## Ghi chú / xử lý sự cố thường gặp (bản Kaggle)

- **Không thấy nút GPU/Internet, giao diện như trống trơn**: bảng cài đặt bên phải Kaggle hay bị thu gọn. Đảm bảo đã **Edit** notebook (không phải đang xem bản đã lưu), rồi bấm **Settings** trên thanh tab trên cùng (Notebook | Input | Output | Logs | Settings). Nếu vẫn không thấy Accelerator/Internet: khả năng cao **chưa xác minh số điện thoại** — vào Settings tài khoản (ảnh đại diện góc phải) → Phone Verification.
- **`CUDA out of memory`**: giảm `BATCH_SIZE` ở Bước 0 (thử 8, rồi 4), chạy lại Bước 5.
- **Train xong nhưng giọng nghe chưa giống / còn lỗi phát âm**: tăng `EXTRA_EPOCHS` rồi chạy lại Bước 5 và Bước 6 — không cần làm lại từ đầu.
- **Câu quá dài bị bỏ qua khi train**: thêm `--max-phoneme-ids 400` vào lệnh train ở Bước 5 nếu log báo drop câu.
- **`AssertionError: Khong tim thay bo du lieu nao` ở Bước 1b**: xem log ngay phía trên assert đó — nó liệt kê toàn bộ file đã lấy được. Hai nguyên nhân hay gặp nhất: (1) dataset được nén/upload từ thư mục project thay vì đúng thư mục `dataset` cạnh file `.exe` đã build, (2) chưa thu được câu nào thành công trong app (kiểm tra dòng "Đã thu: X" > 0 trong VoiceRecorder trước khi nén).
- **`/kaggle/working` đầy dung lượng (~19.5GB mỗi phiên)**: dùng "Bước 5b" để dọn bớt checkpoint cũ, chỉ giữ vài bản mới nhất.
- **Mở lại notebook / phiên mới**: mọi thứ trong `/kaggle/working` của phiên cũ đều mất sạch trừ khi đã **Commit**. Luôn chạy lại **từ Bước 0** theo đúng thứ tự — dùng `Run → Run All` cho nhanh.
- Notebook cố tình dùng repo `rhasspy/piper` (đã archived nhưng vẫn chạy tốt) thay vì `OHF-Voice/piper1-gpl` mới hơn, vì mới hơn hiện chưa nạp ổn định checkpoint `vais1000-medium` cũ.